# MIDI Dataset Downloader → Dropbox

This notebook downloads MIDI datasets and uploads them directly to your Dropbox.

**Datasets:**
- Groove MIDI Dataset (E-GMD) from Google Drive
- MAESTRO v3.0.0 (MIDI-only) from Google Storage
- ComMU Dataset from POZAlabs

**License Notes:**
- Groove MIDI: CC-BY 4.0 (Commercial OK)
- MAESTRO: CC-BY-NC-SA 4.0 (Non-commercial only)
- ComMU: CC-BY-NC-SA 4.0 (Non-commercial only)

## Step 1: Install Dependencies

In [ ]:
!pip install dropbox gdown tqdm

## Step 2: Configure Dropbox

To get your Dropbox access token:
1. Go to https://www.dropbox.com/developers/apps
2. Click "Create app"
3. Choose "Scoped access" → "Full Dropbox"
4. Name it something like "MIDI-Downloader"
5. Go to the "Permissions" tab and enable `files.content.write` and `files.content.read`
6. Go to "Settings" tab → Generate access token
7. Paste the token below

In [ ]:
# Configuration
DROPBOX_ACCESS_TOKEN = "sl.u.AGPmFQpzz7ORKF76L76LaFqWA2Sf_KCsyeNxVLJeFaJPnQ8yAYIa65cAQ9t9YASi7MWiDx-qxtDIw3JKmHAhXEAUxvC5WlajXFp_NZdEN-VJ83qC0tR_2_vCFK_RnjOWHf4Bj4eIB5W7KHq4MemVsSyhZW3qpRuz154z8d5EBSj0mOgkKCGTxhFGV739xdrkHfTElRT6yAqzV7FhorkwTOyZg-8GhUB-M0aK9BWRMQb2Q5TwMfWSNEw--uRlZ72DeTufpQO7EGOp1ENC0FvEfpBEkpVy-hSfdoSxby1zBNbvcshzNcl4M_OIoIZNpAdPF06WkA3_WXv1shSc0pxfVP9eBVL2FhrQLHKaf_-ROZbZbKy2YbhlZyG9JEEX8XRuP9fSQgnRRPoq7P-_mdfEbhuKo_tct0ULzggm9Lw5rLjHYavCXX35yHSP3_JXyQHlyTlDcFTmjhzNzOOxPwUuFW5ChQnqgMoc7ddxNIZ1a4ctmL7mTlI1-Woz_USR6H1u64_6czQb-f1XZ4o38jsaN5e0kMgSyMtodxQTquO6relLPDVLxUj6eDFqjUOQu9wqVB5UsHgALclFc16mMHEUX-HrDjim2DrEBd-ax0WxBO4EbtocgDcRyCtVexaOrDPFnlac6vaXLq3mxrS49bjBzVTwNmViF9fbfwCiW8F4ZmpJzFBkL9bxLZiytnnIIYsIeY86CRVl4-7dB398lCEWmS3QvvMISh7mThJ1PmDvcmC-6fRAnpdGu4uuJ4U1T_YEvDg5KhWd3mVZaJeI-TjUr28iPO8VG6qBJAfNHCXEmJ9z4m1cB6wgcrypL73liV-KgHbDePrxdxPfOBAtNYM8AUt8QegWC4v0FO7F4oN3RNU01fHgIsMOLREcmgOJDCRGiyqOXyOq7v4mVZrifCCYFuRWZUwl5Y8__vcGZbECFLoJ2CATeCFG-N4xIv_dQwj24eGsw430iRifPCh7q38abnZj3iq-_fytiHs4kdVSOUYMGDxjg1MpS9r2hoKK1jXrMEkyh8vfOunBNfoN7kTXAl2LowY4F6-I2AwOekcMb_gtz3aWacyqZSWEiBSL20Z77K__3ycn2Yjq0ZoCPgkWChDTvEPjq9kKAOfinzdTZe2Hu3i_2LWwsKaT1_HuKKdEQo7OXIHWd1YS500xyXjaVAU8HyQ05I_Q80jsQRnX81VI547GeDx5WF9tG1fZZpevp6GZ100zyvAnudBSsMocXgKa6h2ER3k7JQqZ4qEIyFx3pS511LU7Q3VlgHdCtmHpPee876x3OhMS06mxgMzoegjKtQa9bnjDtBzuh825n_e5fQ"  # Replace with your token
DROPBOX_DEST_FOLDER = "/midi_datasets"  # Destination folder in Dropbox

# Dataset URLs
DATASETS = {
    "groove_midi": {
        "type": "gdrive_folder",
        "url": "https://drive.google.com/drive/folders/1Stz3CAvMoplo79LR5I3onMWRelCugBYS",
        "folder_id": "1Stz3CAvMoplo79LR5I3onMWRelCugBYS",
        "output_name": "groove_midi"
    },
    "maestro": {
        "type": "direct",
        "url": "https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip",
        "output_name": "maestro-v3.0.0-midi.zip"
    },
    "commu": {
        "type": "direct",
        "url": "https://pozalabs.github.io/ComMU/assets/ComMU.tar",
        "output_name": "ComMU.tar"
    }
}

## Step 3: Helper Functions

In [ ]:
import os
import dropbox
import gdown
import requests
from tqdm import tqdm
import shutil

def init_dropbox(token):
    """Initialize Dropbox client."""
    dbx = dropbox.Dropbox(token)
    # Verify connection
    try:
        account = dbx.users_get_current_account()
        print(f"Connected to Dropbox as: {account.name.display_name}")
        return dbx
    except Exception as e:
        print(f"Error connecting to Dropbox: {e}")
        return None

def download_direct(url, output_path):
    """Download file from direct URL with progress bar."""
    print(f"Downloading from: {url}")
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    
    with open(output_path, 'wb') as f:
        with tqdm(total=total_size, unit='B', unit_scale=True, desc=output_path) as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                pbar.update(len(chunk))
    
    print(f"Downloaded: {output_path} ({os.path.getsize(output_path) / 1e6:.1f} MB)")
    return output_path

def download_gdrive_folder(folder_id, output_dir):
    """Download entire Google Drive folder."""
    print(f"Downloading Google Drive folder: {folder_id}")
    os.makedirs(output_dir, exist_ok=True)
    gdown.download_folder(id=folder_id, output=output_dir, quiet=False)
    
    # Zip the folder for easier upload
    zip_path = f"{output_dir}.zip"
    print(f"Zipping folder to: {zip_path}")
    shutil.make_archive(output_dir, 'zip', output_dir)
    
    # Clean up the folder
    shutil.rmtree(output_dir)
    
    print(f"Created: {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")
    return zip_path

def upload_to_dropbox(dbx, local_path, dropbox_path, chunk_size=50 * 1024 * 1024):
    """Upload file to Dropbox with chunked upload for large files."""
    file_size = os.path.getsize(local_path)
    print(f"Uploading {local_path} ({file_size / 1e6:.1f} MB) to Dropbox...")
    
    with open(local_path, 'rb') as f:
        if file_size <= chunk_size:
            # Small file - single upload
            dbx.files_upload(f.read(), dropbox_path, mode=dropbox.files.WriteMode.overwrite)
        else:
            # Large file - chunked upload
            upload_session = dbx.files_upload_session_start(f.read(chunk_size))
            cursor = dropbox.files.UploadSessionCursor(
                session_id=upload_session.session_id,
                offset=f.tell()
            )
            commit = dropbox.files.CommitInfo(path=dropbox_path, mode=dropbox.files.WriteMode.overwrite)
            
            with tqdm(total=file_size, initial=chunk_size, unit='B', unit_scale=True, desc="Uploading") as pbar:
                while f.tell() < file_size:
                    remaining = file_size - f.tell()
                    if remaining <= chunk_size:
                        dbx.files_upload_session_finish(f.read(remaining), cursor, commit)
                        pbar.update(remaining)
                    else:
                        dbx.files_upload_session_append_v2(f.read(chunk_size), cursor)
                        cursor.offset = f.tell()
                        pbar.update(chunk_size)
    
    print(f"Uploaded to Dropbox: {dropbox_path}")
    return True

## Step 4: Initialize Dropbox Connection

In [ ]:
# Initialize Dropbox
dbx = init_dropbox(DROPBOX_ACCESS_TOKEN)

if dbx is None:
    raise Exception("Failed to connect to Dropbox. Check your access token.")

## Step 5: Download and Upload Datasets

Run each cell below to download and upload each dataset individually, or run them all.

### 5a. Groove MIDI Dataset (Google Drive)

In [ ]:
# Download Groove MIDI from Google Drive
dataset = DATASETS["groove_midi"]
local_path = download_gdrive_folder(dataset["folder_id"], dataset["output_name"])

# Upload to Dropbox
dropbox_path = f"{DROPBOX_DEST_FOLDER}/groove_midi.zip"
upload_to_dropbox(dbx, local_path, dropbox_path)

# Clean up local file
os.remove(local_path)
print(f"Cleaned up local file: {local_path}")

### 5b. MAESTRO Dataset (Direct Download)

In [ ]:
# Download MAESTRO
dataset = DATASETS["maestro"]
local_path = download_direct(dataset["url"], dataset["output_name"])

# Upload to Dropbox
dropbox_path = f"{DROPBOX_DEST_FOLDER}/{dataset['output_name']}"
upload_to_dropbox(dbx, local_path, dropbox_path)

# Clean up local file
os.remove(local_path)
print(f"Cleaned up local file: {local_path}")

### 5c. ComMU Dataset (Direct Download)

In [ ]:
# Download ComMU
dataset = DATASETS["commu"]
local_path = download_direct(dataset["url"], dataset["output_name"])

# Upload to Dropbox
dropbox_path = f"{DROPBOX_DEST_FOLDER}/{dataset['output_name']}"
upload_to_dropbox(dbx, local_path, dropbox_path)

# Clean up local file
os.remove(local_path)
print(f"Cleaned up local file: {local_path}")

## Step 6: Verify Uploads

In [ ]:
# List files in the Dropbox destination folder
print(f"\nFiles in Dropbox {DROPBOX_DEST_FOLDER}:")
print("-" * 50)

try:
    result = dbx.files_list_folder(DROPBOX_DEST_FOLDER)
    for entry in result.entries:
        if isinstance(entry, dropbox.files.FileMetadata):
            size_mb = entry.size / 1e6
            print(f"  {entry.name} ({size_mb:.1f} MB)")
        else:
            print(f"  {entry.name}/")
except dropbox.exceptions.ApiError as e:
    print(f"Error listing folder: {e}")

print("\nAll datasets downloaded and uploaded successfully!")

## Optional: Download All at Once

Run this cell to download and upload all datasets in sequence.

In [ ]:
# Download and upload all datasets
for name, dataset in DATASETS.items():
    print(f"\n{'='*60}")
    print(f"Processing: {name}")
    print(f"{'='*60}")
    
    try:
        # Download based on type
        if dataset["type"] == "gdrive_folder":
            local_path = download_gdrive_folder(dataset["folder_id"], dataset["output_name"])
            dropbox_filename = f"{dataset['output_name']}.zip"
        else:
            local_path = download_direct(dataset["url"], dataset["output_name"])
            dropbox_filename = dataset["output_name"]
        
        # Upload to Dropbox
        dropbox_path = f"{DROPBOX_DEST_FOLDER}/{dropbox_filename}"
        upload_to_dropbox(dbx, local_path, dropbox_path)
        
        # Clean up
        os.remove(local_path)
        print(f"Completed: {name}")
        
    except Exception as e:
        print(f"Error processing {name}: {e}")
        continue

print(f"\n{'='*60}")
print("All downloads complete!")
print(f"{'='*60}")